# Model: mistralai/Mistral-7B-Instruct-v0.3

## 🚀 RMIT Hackathon 2026 — Challenge 1: Multilingual Prompt-Attack Classification
> **Notebook Naming:** `RMIT_SGS_BakhiBachKhoa_C1`
> **BTC Strict Rule:** Strictly ONE approved model backbone (`mistralai/Mistral-7B-Instruct-v0.3`). No multi-backbone ensembling or blending.
> **Architecture:** QLoRA 4-bit NF4 (`double_quant=True`, `compute_dtype=torch.float16`) + `MistralForSequenceClassification` (`num_labels=3`) + Head/Tail Instruction-Preserving Truncation.

---

### 📖 Hướng Dẫn Chạy 3 Bước Trên Kaggle (T4 x2)
1. **Bước 1 — Smoke Test (~10–15 phút):**
   - Add Input trên Kaggle: Dataset cuộc thi + `folds.csv` + Model `aisuko/mistral-7b-instruct-v0.3` (Transformers / default).
   - Giữ `SMOKE_TEST = True` ở Cell 1 $\rightarrow$ Chạy toàn bộ notebook trên Interactive Session (GPU T4 x2).
   - Kiểm tra log cuối Cell 3: `import peft/bitsandbytes OK`, `peak_vram_gb < 14.0 GB`, `tokens/sec`, và `ETA 1 fold`.
   - Nếu `ETA 1 fold > 90 phút`: hạ `MAX_LEN = 384` hoặc đổi `MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"` (Add model `aisuko/granite-3.3-2b-instruct`).
2. **Bước 2 — Chạy Fold 0 & Fold 1 song song trên Tài khoản 1 (~1–1.5 giờ):**
   - Đặt `SMOKE_TEST = False`, `FOLDS_TO_RUN = {"0": [0], "1": [1]}` ở Cell 1 $\rightarrow$ Bấm **Save Version (Save & Run All)**.
   - Kiểm tra Gate 12:45: Nếu Fold 0 đạt `Val Macro-F1 >= 0.975` và cụm `pt` 500–2000 ký tự cải thiện rõ $\rightarrow$ qua Bước 3.
3. **Bước 3 — Chạy Fold 2, 3, 4 trên Tài khoản 2/3 & Gộp Kết Quả:**
   - Tài khoản 2: `SMOKE_TEST = False`, `FOLDS_TO_RUN = {"0": [2], "1": [3]}`.
   - Tài khoản 3 (hoặc chạy tiếp): `SMOKE_TEST = False`, `FOLDS_TO_RUN = {"0": [4], "1": []}`.
   - Khi ghép đủ 5 fold: gắn output của các lần chạy vào `RESUME_DIRS = ["/kaggle/input/..."]` để Cell 5 tự động gộp đủ 5 fold, tối ưu multiplier và xuất `submission.csv`.


In [ ]:
# =============================================================================
# ⚙️ 1. CẤU HÌNH SIÊU THAM SỐ & KIỂM TRA MÔI TRƯỜNG PEFT / BITSANDBYTES
# =============================================================================
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

# Kiểm tra peft & bitsandbytes; nếu thiếu và có Internet hoặc wheel local thì tự cài
def ensure_qlora_packages() -> None:
    missing = [pkg for pkg in ("peft", "bitsandbytes", "accelerate") if importlib.util.find_spec(pkg) is None]
    if missing:
        print(f"[!] Phát hiện thiếu thư viện: {missing}. Đang thử cài đặt qua pip...", flush=True)
        # Tìm wheel offline trong /kaggle/input nếu có
        wheel_dirs = [str(p.parent) for p in Path("/kaggle/input").rglob("*.whl")] if Path("/kaggle/input").exists() else []
        if wheel_dirs:
            cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", wheel_dirs[0], *missing]
        else:
            cmd = [sys.executable, "-m", "pip", "install", "-q", "-U", *missing]
        subprocess.run(cmd, check=False)

    import peft
    import bitsandbytes as bnb
    print(f"[+] import peft/bitsandbytes OK! (peft={peft.__version__}, bitsandbytes={bnb.__version__})", flush=True)

try:
    ensure_qlora_packages()
except Exception as e:
    print(f"[!] Cảnh báo môi trường QLoRA: {e} (Hãy bật Internet trên Kaggle hoặc gắn dataset chứa wheel peft/bitsandbytes)", flush=True)

# Thông tin đội thi & Backbone chính thức (thuộc Whitelist BTC)
TEAM_NAME = "BakhiBachKhoa"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
# Fallback sẵn sàng nếu ETA Mistral > 90 phút/fold (chỉ cần bỏ comment dòng dưới):
# MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"

# CHẾ ĐỘ CHẠY:
# - SMOKE_TEST = True: Chạy thử nhanh trên Fold 0 (256 train, 128 val, 64 test, 30 opt steps) để đo VRAM & ETA.
# - SMOKE_TEST = False: Chạy chính thức các fold trong FOLDS_TO_RUN.
SMOKE_TEST = True
SUBMIT_MODE = "fold_avg"

# Phân bổ GPU cho nhiều tài khoản chạy song song:
# - Tài khoản 1: {"0": [0], "1": [1]}
# - Tài khoản 2: {"0": [2], "1": [3]}
# - Tài khoản 3: {"0": [4], "1": []}
FOLDS_TO_RUN = {"0": [0], "1": [1]}
GPU_FOLDS = FOLDS_TO_RUN

# Siêu tham số QLoRA chuẩn cho T4 x2 (16GB VRAM / GPU)
MAX_LEN = 512           # Mặc định 512 (cho phép hạ xuống 384 nếu muốn tăng tốc ~25%)
EPOCHS = 1              # 1 epoch cho LLM 7B (eval 2 lần/epoch: giữa epoch và cuối epoch)
LR = 1e-4               # Learning rate cho QLoRA
BATCH_SIZE = 4          # Micro-batch size trên mỗi T4
GRAD_ACCUM = 4          # Effective batch size = 4 * 4 = 16
WARMUP_RATIO = 0.05     # Warmup 5% tổng số steps
WEIGHT_DECAY = 0.01
ADAM_EPS = 1e-6

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
CLASS_WEIGHTS = [1.0, 2.0, 8.8]
SEED = 42
MAX_TRAIN_MINUTES = 660  # Guard an toàn (11 giờ) trước giới hạn 12 giờ của Kaggle

OUT_DIR = "/kaggle/working"
RESUME_DIRS = []         # Danh sách các thư mục Kaggle Input chứa *_fold*.* từ các tài khoản/version trước
RESUME_DIR = None

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(f"{OUT_DIR}/logs").mkdir(parents=True, exist_ok=True)

CONFIG_DATA = {
    "TEAM_NAME": TEAM_NAME,
    "MODEL_NAME": MODEL_NAME,
    "SMOKE_TEST": SMOKE_TEST,
    "SUBMIT_MODE": SUBMIT_MODE,
    "FOLDS_TO_RUN": FOLDS_TO_RUN,
    "GPU_FOLDS": GPU_FOLDS,
    "MAX_LEN": MAX_LEN,
    "EPOCHS": EPOCHS,
    "LR": LR,
    "BATCH_SIZE": BATCH_SIZE,
    "GRAD_ACCUM": GRAD_ACCUM,
    "WARMUP_RATIO": WARMUP_RATIO,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "ADAM_EPS": ADAM_EPS,
    "LORA_R": LORA_R,
    "LORA_ALPHA": LORA_ALPHA,
    "LORA_DROPOUT": LORA_DROPOUT,
    "CLASS_WEIGHTS": CLASS_WEIGHTS,
    "SEED": SEED,
    "MAX_TRAIN_MINUTES": MAX_TRAIN_MINUTES,
    "OUT_DIR": OUT_DIR,
    "RESUME_DIRS": RESUME_DIRS,
    "RESUME_DIR": RESUME_DIR,
}

config_path = Path(OUT_DIR) / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG_DATA, f, indent=2, ensure_ascii=False)

print(f"[*] Đã ghi cấu hình vào: {config_path}", flush=True)
print(f"[*] Model: {MODEL_NAME} | SMOKE_TEST: {SMOKE_TEST} | FOLDS_TO_RUN: {FOLDS_TO_RUN}", flush=True)
print(f"[*] MAX_LEN: {MAX_LEN} | Batch: {BATCH_SIZE}x{GRAD_ACCUM}=16 | LR: {LR} | LoRA r={LORA_R}, alpha={LORA_ALPHA}", flush=True)


In [ ]:
%%writefile train_llm.py
"""
train_llm.py
Script huấn luyện K-Fold QLoRA 4-bit NF4 cho mistralai/Mistral-7B-Instruct-v0.3
(và fallback ibm-granite/granite-3.3-2b-instruct) trên Kaggle GPU T4.
Tuân thủ nghiêm ngặt luật Single-Model của RMIT Hackathon 2026.
"""

import argparse
import gc
import json
import math
import os
import random
import shutil
import sys
import time
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Sampler
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    MistralForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cudnn.benchmark = False

VALID_LABELS = ("BENIGN", "JAILBREAK", "PROMPT_INJECTION")
LABEL2ID = {"BENIGN": 0, "JAILBREAK": 1, "PROMPT_INJECTION": 2}
ID2LABEL = {0: "BENIGN", 1: "JAILBREAK", 2: "PROMPT_INJECTION"}
INSTRUCTION_TEXT = "Classify the following prompt as BENIGN, JAILBREAK or PROMPT_INJECTION.\n"


def set_seed(seed: int) -> None:
    """Cố định seed cho random, numpy và torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def log_msg(msg: str, log_file: Optional[Path] = None) -> None:
    """In log ra stdout với flush=True và ghi vào file log."""
    print(msg, flush=True)
    if log_file is not None:
        try:
            with open(log_file, "a", encoding="utf-8") as f:
                f.write(msg + "\n")
        except Exception:
            pass


def find_model_dir(model_name: str) -> Path:
    """
    Tìm thư mục chứa backbone LLM trên Kaggle Models (/kaggle/input/...).
    Hỗ trợ mistralai/Mistral-7B-Instruct-v0.3 và fallback ibm-granite/granite-3.3-2b-instruct.
    """
    target_lower = model_name.lower()
    is_mistral = "mistral" in target_lower
    is_granite = "granite" in target_lower

    search_dirs = [Path("/kaggle/input"), Path("models"), Path(".")]
    found_configs = []
    for base in search_dirs:
        if base.exists():
            for cfg in sorted(base.rglob("config.json")):
                if cfg.parent == Path("/kaggle/working"):
                    continue
                found_configs.append(str(cfg))
                cfg_str = str(cfg).lower()
                try:
                    data = json.loads(cfg.read_text(encoding="utf-8"))
                    mtype = str(data.get("model_type", "")).lower()
                except Exception:
                    mtype = ""

                if is_mistral and ("mistral" in cfg_str or mtype == "mistral"):
                    return cfg.parent
                if is_granite and ("granite" in cfg_str or mtype in ("granite", "llama")):
                    return cfg.parent

    raise FileNotFoundError(
        f"Không tìm thấy model '{model_name}'! Vui lòng Add Input trên Kaggle Models "
        f"(aisuko/mistral-7b-instruct-v0.3 hoặc aisuko/granite-3.3-2b-instruct). "
        f"Các config.json tìm thấy: {found_configs}"
    )


def find_file(name: str) -> Path:
    """Tìm file dữ liệu bằng rglob."""
    search_dirs = [Path("/kaggle/input"), Path("data"), Path(".")]
    for base in search_dirs:
        if base.exists():
            for f in sorted(base.rglob(name)):
                if f.is_file():
                    return f
    raise FileNotFoundError(f"Không tìm thấy tệp bắt buộc: '{name}'")


def clean_text_safety(text: Any) -> str:
    """Chuẩn hóa NFC, loại bỏ null byte, chuẩn hóa xuống dòng, giữ nguyên dấu Romance."""
    if not isinstance(text, str):
        if pd.isna(text) or text is None:
            return ""
        text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text.strip()


def setup_tokenizer_and_pad(model_dir: Path) -> Tuple[Any, int]:
    """
    Khởi tạo tokenizer cho Mistral/Granite:
    - Dùng tokenizer.unk_token (id 0) làm pad_token.
    - KHÔNG dùng eos_token làm pad, KHÔNG resize embedding.
    - Padding bên phải (padding_side = 'right'), không tự thêm eos vào cuối input.
    """
    try:
        tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=True)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=False)

    tokenizer.padding_side = "right"
    if hasattr(tokenizer, "add_eos_token"):
        tokenizer.add_eos_token = False

    assert tokenizer.unk_token is not None and tokenizer.unk_token_id is not None, (
        "Tokenizer không có unk_token!"
    )
    tokenizer.pad_token = tokenizer.unk_token
    tokenizer.pad_token_id = tokenizer.unk_token_id
    pad_id = int(tokenizer.pad_token_id)
    assert pad_id != tokenizer.eos_token_id, "CẤM dùng eos_token_id làm pad_token_id!"
    return tokenizer, pad_id


def prepare_llm_input_ids(
    bodies: List[str],
    tokenizer: Any,
    max_len: int,
    pad_id: int,
) -> List[List[int]]:
    """
    Tokenize văn bản với cấu trúc:
      [BOS (nếu có)] + Instruction + Body ('lang: {language} | {clean_text}')
    - Không tự động thêm EOS vào cuối.
    - Instruction luôn được giữ nguyên.
    - Nếu dài quá MAX_LEN: cắt (MAX_LEN - len(prefix)) // 2 token đầu của Body + phần còn lại ở cuối Body.
    - Đảm bảo không có token nào bên trong chuỗi trùng với pad_id (id 0) để pooling của
      MistralForSequenceClassification luôn lấy chính xác token cuối cùng không phải pad.
    """
    instr_ids = tokenizer.encode(INSTRUCTION_TEXT, add_special_tokens=False)
    prefix_ids: List[int] = []
    if tokenizer.bos_token_id is not None and tokenizer.bos_token_id != pad_id:
        prefix_ids.append(int(tokenizer.bos_token_id))
    prefix_ids.extend(int(x) for x in instr_ids if int(x) != pad_id)

    body_budget = max_len - len(prefix_ids)
    assert body_budget >= 64, f"MAX_LEN={max_len} quá nhỏ so với độ dài instruction ({len(prefix_ids)})"
    head_budget = body_budget // 2
    tail_budget = body_budget - head_budget

    enc_bodies = tokenizer(bodies, add_special_tokens=False, truncation=False)["input_ids"]
    fallback_tok = int(tokenizer.eos_token_id) if tokenizer.eos_token_id is not None else 1

    all_ids: List[List[int]] = []
    for b_ids in enc_bodies:
        # Thay thế nếu có token trùng pad_id (unk=0) bên trong câu để tránh cắt nhầm vị trí pooling
        clean_b = [int(t) if int(t) != pad_id else fallback_tok for t in b_ids]
        if len(clean_b) <= body_budget:
            seq = prefix_ids + clean_b
        else:
            seq = prefix_ids + clean_b[:head_budget] + clean_b[-tail_budget:]
        assert len(seq) <= max_len
        all_ids.append(seq)
    return all_ids


def verify_last_token_pooling(input_ids: torch.Tensor, attention_mask: torch.Tensor, pad_id: int) -> None:
    """
    Kiểm tra toán học công thức tìm last non-pad token của MistralForSequenceClassification:
      sequence_lengths = (torch.eq(input_ids, pad_id).int().argmax(-1) - 1) % input_ids.shape[-1]
    Phải khớp 100% với (attention_mask.sum(dim=-1) - 1).
    """
    seq_lens_hf = (torch.eq(input_ids, pad_id).int().argmax(-1) - 1) % input_ids.shape[-1]
    seq_lens_true = attention_mask.sum(dim=-1) - 1
    if not torch.equal(seq_lens_hf, seq_lens_true):
        raise AssertionError(
            f"Lỗi pooling vị trí token cuối! HF={seq_lens_hf.tolist()} vs True={seq_lens_true.tolist()}"
        )


class SmartBatchSampler(Sampler):
    """Smart Batching: chia mega-batch -> sort theo độ dài -> chia mini-batch -> trộn mini-batch."""
    def __init__(
        self,
        lengths: List[int],
        batch_size: int,
        mega_batch_mult: int = 64,
        shuffle: bool = True,
        seed: int = 42,
    ):
        super().__init__()
        self.lengths = lengths
        self.batch_size = batch_size
        self.mega_batch_size = batch_size * mega_batch_mult
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def __iter__(self):
        rng = np.random.RandomState(self.seed + self.epoch)
        indices = np.arange(len(self.lengths))
        if self.shuffle:
            rng.shuffle(indices)

        batches = []
        for i in range(0, len(indices), self.mega_batch_size):
            mega_chunk = indices[i:i + self.mega_batch_size]
            sorted_mega = [int(x) for x in sorted(mega_chunk, key=lambda idx: self.lengths[int(idx)])]
            for j in range(0, len(sorted_mega), self.batch_size):
                batches.append(sorted_mega[j:j + self.batch_size])

        if self.shuffle:
            batch_order = rng.permutation(len(batches))
            batches = [batches[k] for k in batch_order]

        for b in batches:
            yield b

    def __len__(self) -> int:
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch


class TextDataset(Dataset):
    def __init__(self, input_ids: List[List[int]], labels: Optional[List[int]] = None):
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = {"input_ids": self.input_ids[idx]}
        if self.labels is not None:
            item["label"] = self.labels[idx]
        return item


def make_collate_fn(pad_id: int):
    """Dynamic right-padding tới độ dài lớn nhất trong mini-batch."""
    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_len = max(len(x["input_ids"]) for x in batch)
        bs = len(batch)
        input_ids = torch.full((bs, batch_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((bs, batch_len), dtype=torch.long)
        has_labels = "label" in batch[0]
        labels = []

        for i, item in enumerate(batch):
            seq = item["input_ids"]
            seq_len = len(seq)
            input_ids[i, :seq_len] = torch.tensor(seq, dtype=torch.long)
            attention_mask[i, :seq_len] = 1
            if has_labels:
                labels.append(item["label"])

        res = {"input_ids": input_ids, "attention_mask": attention_mask}
        if has_labels:
            res["labels"] = torch.tensor(labels, dtype=torch.long)
        return res
    return collate_fn


def build_qlora_model(
    model_dir: Path,
    config: Dict[str, Any],
    pad_id: int,
    device: torch.device,
    log_file: Optional[Path] = None,
) -> nn.Module:
    """
    Khởi tạo MistralForSequenceClassification (hoặc AutoModelForSequenceClassification cho Granite)
    với BitsAndBytes 4-bit NF4, double quant, compute_dtype=float16, gradient checkpointing
    và LoRA target q,k,v,o,gate,up,down + modules_to_save=['score'] (head score ở fp32).
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    hf_cfg = AutoConfig.from_pretrained(
        str(model_dir),
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    hf_cfg.pad_token_id = pad_id
    hf_cfg.use_cache = False

    model_cls = (
        MistralForSequenceClassification
        if getattr(hf_cfg, "model_type", "") == "mistral"
        else AutoModelForSequenceClassification
    )

    device_index = device.index if device.index is not None else 0
    model = model_cls.from_pretrained(
        str(model_dir),
        config=hf_cfg,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": device_index},
        low_cpu_mem_usage=True,
    )
    model.config.pad_token_id = pad_id
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    lora_config = LoraConfig(
        r=int(config["LORA_R"]),
        lora_alpha=int(config["LORA_ALPHA"]),
        lora_dropout=float(config["LORA_DROPOUT"]),
        bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        modules_to_save=["score"],
    )
    model = get_peft_model(model, lora_config)

    # Đảm bảo head 'score' và các tham số trainable ở fp32 để ổn định gradient dưới FP16 autocast
    score_fp32_count = 0
    for name, param in model.named_parameters():
        if param.requires_grad or "score" in name:
            param.data = param.data.to(torch.float32)
            if "score" in name:
                score_fp32_count += param.numel()
    assert score_fp32_count > 0, "Không tìm thấy head 'score' ở fp32!"

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    log_msg(
        f"[*] QLoRA khởi tạo thành công: Trainable params = {trainable_params:,} / {all_params:,} "
        f"({100.0 * trainable_params / all_params:.2f}%) | score head fp32 = {score_fp32_count} params",
        log_file,
    )
    return model


@torch.no_grad()
def predict_dataset(
    model: nn.Module,
    input_ids_list: List[List[int]],
    pad_id: int,
    batch_size: int = 8,
    device: torch.device = torch.device("cuda"),
) -> np.ndarray:
    """Dự đoán xác suất với sort theo độ dài để tăng tốc và khôi phục đúng thứ tự gốc."""
    model.eval()
    n_samples = len(input_ids_list)
    sorted_indices = sorted(range(n_samples), key=lambda i: len(input_ids_list[i]))

    sorted_probs_list = []
    for start_idx in range(0, n_samples, batch_size):
        chunk_indices = sorted_indices[start_idx:start_idx + batch_size]
        chunk_seqs = [input_ids_list[i] for i in chunk_indices]
        max_len = max(len(seq) for seq in chunk_seqs)
        bs = len(chunk_seqs)

        input_ids_cpu = torch.full((bs, max_len), pad_id, dtype=torch.long)
        attention_mask_cpu = torch.zeros((bs, max_len), dtype=torch.long)
        for row, seq in enumerate(chunk_seqs):
            seq_len = len(seq)
            input_ids_cpu[row, :seq_len] = torch.tensor(seq, dtype=torch.long)
            attention_mask_cpu[row, :seq_len] = 1

        if start_idx == 0:
            verify_last_token_pooling(input_ids_cpu, attention_mask_cpu, pad_id)

        input_ids = input_ids_cpu.to(device, non_blocking=True)
        attention_mask = attention_mask_cpu.to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        sorted_probs_list.append(probs)

    sorted_probs = np.vstack(sorted_probs_list)
    orig_probs = np.zeros_like(sorted_probs, dtype=np.float32)
    orig_probs[sorted_indices] = sorted_probs
    assert orig_probs.shape == (n_samples, 3)
    return orig_probs


def save_lora_checkpoint(model: nn.Module, ckpt_path: Path, adapter_dir: Path) -> None:
    """Lưu trọng số trainable (LoRA + score head) vào /tmp."""
    trainable_state = {
        k: v.detach().cpu().clone()
        for k, v in model.named_parameters()
        if v.requires_grad
    }
    torch.save(trainable_state, ckpt_path)
    try:
        model.save_pretrained(str(adapter_dir))
    except Exception:
        pass


def load_lora_checkpoint(model: nn.Module, ckpt_path: Path) -> None:
    """Khôi phục trọng số trainable (LoRA + score head) tốt nhất từ /tmp."""
    saved_state = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    for name, param in model.named_parameters():
        if name in saved_state:
            param.data.copy_(saved_state[name].to(device=param.device, dtype=param.dtype))


def train_single_fold(
    fold: int,
    config: Dict[str, Any],
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    train_input_ids: List[List[int]],
    test_input_ids: List[List[int]],
    model_dir: Path,
    pad_id: int,
    device: torch.device,
    script_t0: float,
    is_smoke: bool = False,
    out_dir: Path = Path("/kaggle/working"),
    log_file: Optional[Path] = None,
) -> None:
    log_msg(f"\n{'='*70}\n>>> [BẮT ĐẦU FOLD {fold}] {'(SMOKE TEST)' if is_smoke else ''}\n{'='*70}", log_file)

    fold_seed = int(config["SEED"]) + int(fold)
    set_seed(fold_seed)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    oof_save_path = out_dir / f"oof_fold{fold}.npy"
    test_save_path = out_dir / f"test_fold{fold}.npy"
    val_idx_save_path = out_dir / f"val_idx_fold{fold}.npy"
    metrics_save_path = out_dir / f"metrics_fold{fold}.json"

    if not is_smoke and oof_save_path.exists() and test_save_path.exists():
        log_msg(f"[*] Fold {fold} đã có sẵn file kết quả. BỎ QUA (Resume)!", log_file)
        return

    val_mask = (train_df["fold"] == fold).values
    val_indices = np.where(val_mask)[0]
    train_indices = np.where(~val_mask)[0]

    if is_smoke:
        train_indices = train_indices[:256]
        val_indices = val_indices[:128]
        test_input_ids_sub = test_input_ids[:64]
    else:
        test_input_ids_sub = test_input_ids

    train_fold_ids = [train_input_ids[i] for i in train_indices]
    train_fold_labels = [int(train_df["label_id"].iloc[i]) for i in train_indices]
    val_fold_ids = [train_input_ids[i] for i in val_indices]
    val_fold_labels = np.array([int(train_df["label_id"].iloc[i]) for i in val_indices], dtype=int)

    log_msg(f"[*] Đang nạp backbone QLoRA từ: {model_dir} (seed={fold_seed})", log_file)
    model = build_qlora_model(model_dir, config, pad_id, device, log_file=log_file)

    batch_size = int(config["BATCH_SIZE"])
    grad_accum = int(config["GRAD_ACCUM"])
    collate_fn = make_collate_fn(pad_id)

    lengths = [len(seq) for seq in train_fold_ids]
    sampler = SmartBatchSampler(lengths, batch_size=batch_size, shuffle=True, seed=fold_seed)
    train_dataset = TextDataset(train_fold_ids, train_fold_labels)
    train_loader = DataLoader(train_dataset, batch_sampler=sampler, collate_fn=collate_fn, num_workers=0)

    # Phân nhóm tham số trainable cho AdamW
    no_decay = ("bias", "layernorm.weight", "layer_norm.weight", "norm.weight")
    decay_params = [p for n, p in model.named_parameters() if p.requires_grad and not any(nd in n.lower() for nd in no_decay)]
    nodecay_params = [p for n, p in model.named_parameters() if p.requires_grad and any(nd in n.lower() for nd in no_decay)]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": float(config.get("WEIGHT_DECAY", 0.01))},
            {"params": nodecay_params, "weight_decay": 0.0},
        ],
        lr=float(config["LR"]),
        eps=float(config.get("ADAM_EPS", 1e-6)),
    )

    epochs = int(config["EPOCHS"])
    steps_per_epoch = math.ceil(len(train_loader) / grad_accum)
    total_optimizer_steps = steps_per_epoch * epochs
    warmup_steps = int(total_optimizer_steps * float(config.get("WARMUP_RATIO", 0.05)))

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_optimizer_steps,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    class_weights_tensor = torch.tensor(config["CLASS_WEIGHTS"], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    # Eval 2 lần mỗi epoch (giữa epoch và cuối epoch, kể cả epoch 0)
    eval_schedule = set()
    for ep in range(epochs):
        half_opt = max(1, steps_per_epoch // 2)
        mid_pt = min(len(train_loader) - 1, half_opt * grad_accum - 1)
        eval_schedule.add((ep, mid_pt))
        eval_schedule.add((ep, len(train_loader) - 1))

    ckpt_path = Path(f"/tmp/best_lora_fold{fold}.pt")
    adapter_dir = Path(f"/tmp/best_adapter_fold{fold}")
    if ckpt_path.exists():
        ckpt_path.unlink()

    best_f1 = -1.0
    best_per_class = [0.0, 0.0, 0.0]
    best_step_desc = ""
    n_bad_steps = 0
    t0_fold = time.time()
    step_times: List[float] = []
    step_tokens: List[int] = []
    accum_tokens = 0
    max_train_minutes = float(config.get("MAX_TRAIN_MINUTES", 660.0))

    log_msg(
        f"[*] Train samples: {len(train_fold_ids):,} | Val: {len(val_fold_ids):,} | "
        f"Steps/epoch: {steps_per_epoch} | Total opt steps: {total_optimizer_steps}",
        log_file,
    )

    opt_step_count = 0
    stop_early = False

    for epoch in range(epochs):
        if stop_early:
            break
        sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        opt_step_start = time.perf_counter()
        accum_tokens = 0

        for step, batch in enumerate(train_loader):
            is_accum_step = ((step + 1) % grad_accum == 0) or ((step + 1) == len(train_loader))

            if step == 0 and epoch == 0:
                verify_last_token_pooling(batch["input_ids"], batch["attention_mask"], pad_id)
                log_msg("[+] Kiểm tra pooling last non-pad token: OK!", log_file)

            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            accum_tokens += int(attention_mask.sum().item())

            with torch.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits.float(), labels) / grad_accum

            if not torch.isfinite(loss):
                n_bad_steps += 1
                optimizer.zero_grad(set_to_none=True)
                opt_step_start = time.perf_counter()
                accum_tokens = 0
                continue

            scaler.scale(loss).backward()

            if is_accum_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                scale_after = scaler.get_scale()

                optimizer.zero_grad(set_to_none=True)
                if scale_after >= scale_before:
                    scheduler.step()

                opt_step_count += 1
                step_times.append(time.perf_counter() - opt_step_start)
                step_tokens.append(accum_tokens)
                opt_step_start = time.perf_counter()
                accum_tokens = 0

                if opt_step_count % 50 == 0 and not is_smoke:
                    vram_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3) if torch.cuda.is_available() else 0.0
                    avg_t = float(np.mean(step_times[-20:]))
                    log_msg(
                        f"  [Fold {fold} | OptStep {opt_step_count}/{total_optimizer_steps}] "
                        f"loss={float(loss.item()*grad_accum):.4f} | {avg_t:.2f}s/step | VRAM={vram_gb:.2f}GB",
                        log_file,
                    )

            if is_smoke and opt_step_count >= 30:
                stop_early = True
                break

            # Guard giới hạn 12h của Kaggle
            elapsed_total_min = (time.time() - script_t0) / 60.0
            if not is_smoke and elapsed_total_min >= max_train_minutes:
                log_msg(
                    f"[!] Chạm ngưỡng an toàn MAX_TRAIN_MINUTES={max_train_minutes:.0f} phút "
                    f"(đã chạy {elapsed_total_min:.1f} phút). Dừng sớm để lưu kết quả!",
                    log_file,
                )
                stop_early = True
                break

            if not is_smoke and (epoch, step) in eval_schedule:
                val_probs = predict_dataset(model, val_fold_ids, pad_id, batch_size=8, device=device)
                val_preds = val_probs.argmax(axis=1)
                macro_f1 = float(f1_score(val_fold_labels, val_preds, average="macro", zero_division=0))
                per_class = f1_score(val_fold_labels, val_preds, average=None, zero_division=0).tolist()

                # Tính thêm F1 trên cụm khó pt 500-2000 ký tự B/J để phục vụ Gate 12:45
                val_df_slice = train_df.iloc[val_indices]
                char_lens = val_df_slice["text"].astype(str).str.len().values
                pt_bj_mask = (
                    (val_df_slice["language"].values == "pt")
                    & (char_lens >= 500)
                    & (char_lens <= 2000)
                    & np.isin(val_fold_labels, [0, 1])
                )
                pt_bj_f1 = (
                    float(f1_score(val_fold_labels[pt_bj_mask], val_preds[pt_bj_mask], labels=[0, 1], average="macro", zero_division=0))
                    if pt_bj_mask.any() else 0.0
                )
                pt_bj_err = int((val_fold_labels[pt_bj_mask] != val_preds[pt_bj_mask]).sum()) if pt_bj_mask.any() else 0

                log_msg(
                    f"[Eval Ep {epoch+1} Step {step+1}/{len(train_loader)}] "
                    f"Val Macro-F1: {macro_f1:.4f} | "
                    f"B: {per_class[0]:.4f} | J: {per_class[1]:.4f} | PI: {per_class[2]:.4f} | "
                    f"pt_500_2000_BJ F1: {pt_bj_f1:.4f} (lỗi: {pt_bj_err}/{int(pt_bj_mask.sum())})",
                    log_file,
                )

                if macro_f1 > best_f1:
                    best_f1 = macro_f1
                    best_per_class = per_class
                    best_step_desc = f"epoch_{epoch+1}_step_{step+1}"
                    save_lora_checkpoint(model, ckpt_path, adapter_dir)

                model.train()
                opt_step_start = time.perf_counter()

    if is_smoke:
        smoke_val_probs = predict_dataset(model, val_fold_ids, pad_id, batch_size=8, device=device)
        smoke_test_probs = predict_dataset(model, test_input_ids_sub, pad_id, batch_size=8, device=device)
        assert smoke_val_probs.shape == (len(val_fold_ids), 3)
        assert smoke_test_probs.shape == (len(test_input_ids_sub), 3)

        peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 3) if torch.cuda.is_available() else 0.0
        valid_times = step_times[1:] if len(step_times) > 1 else step_times
        valid_tokens = step_tokens[1:] if len(step_tokens) > 1 else step_tokens
        sec_per_step = float(np.mean(valid_times)) if valid_times else 0.0
        tok_per_sec = float(np.sum(valid_tokens) / np.sum(valid_times)) if (valid_times and np.sum(valid_times) > 0) else 0.0

        full_steps_per_fold = math.ceil(11200 / (batch_size * grad_accum)) * epochs
        # Cộng ~12% thời gian cho 2 lần Eval + Predict Test trên 1 fold
        eta_1_fold_min = (sec_per_step * full_steps_per_fold * 1.12) / 60.0

        log_msg(f"\n{'='*70}", log_file)
        log_msg("[SMOKE TEST QLoRA THÀNH CÔNG — BÁO CÁO HIỆU NĂNG]", log_file)
        log_msg("- import peft/bitsandbytes: OK", log_file)
        log_msg(f"- Backbone: {config['MODEL_NAME']} (MAX_LEN={config['MAX_LEN']})", log_file)
        log_msg(f"- VRAM Đỉnh (peak_vram_gb): {peak_vram:.2f} GB (Ngưỡng an toàn T4 16GB: < 14.0 GB)", log_file)
        log_msg(f"- Tốc độ mỗi optimizer step (sec_per_step): {sec_per_step:.3f} giây", log_file)
        log_msg(f"- Thông lượng xử lý (tokens/sec): {tok_per_sec:.1f} tokens/giây", log_file)
        log_msg(f"- Số optimizer steps mỗi fold ({epochs} epoch): {full_steps_per_fold}", log_file)
        log_msg(f"- ETA dự kiến cho 1 fold đầy đủ (bao gồm eval + test): {eta_1_fold_min:.1f} phút ({eta_1_fold_min/60.0:.2f} giờ)", log_file)
        log_msg(f"- Số step lỗi NaN/Inf (n_bad_steps): {n_bad_steps}", log_file)
        if eta_1_fold_min > 90.0:
            log_msg("[!] KHUYẾN NGHỊ: ETA > 90 phút/fold -> Hãy giảm MAX_LEN=384 hoặc chuyển sang ibm-granite/granite-3.3-2b-instruct!", log_file)
        else:
            log_msg("[+] KHUYẾN NGHỊ: ETA <= 90 phút/fold đạt chuẩn! Sẵn sàng đổi SMOKE_TEST=False để train Fold 0 & 1.", log_file)
        log_msg(f"{'='*70}\n", log_file)

        del model, optimizer, scheduler, scaler
        gc.collect()
        torch.cuda.empty_cache()
        return

    log_msg(f"[*] Đang khôi phục adapter LoRA tốt nhất ({best_step_desc}, Macro-F1: {best_f1:.4f})...", log_file)
    if ckpt_path.exists():
        load_lora_checkpoint(model, ckpt_path)

    log_msg(f"[*] Đang sinh OOF probabilities cho Fold {fold}...", log_file)
    oof_probs = predict_dataset(model, val_fold_ids, pad_id, batch_size=8, device=device)
    if best_f1 < 0.0:
        val_preds = oof_probs.argmax(axis=1)
        best_f1 = float(f1_score(val_fold_labels, val_preds, average="macro", zero_division=0))
        best_per_class = f1_score(val_fold_labels, val_preds, average=None, zero_division=0).tolist()
        best_step_desc = "early_stop_final"

    log_msg(f"[*] Đang sinh Test probabilities cho Fold {fold}...", log_file)
    test_probs = predict_dataset(model, test_input_ids_sub, pad_id, batch_size=8, device=device)

    np.save(oof_save_path, oof_probs.astype(np.float32))
    np.save(val_idx_save_path, val_indices.astype(np.int64))
    np.save(test_save_path, test_probs.astype(np.float32))

    fold_minutes = (time.time() - t0_fold) / 60.0
    peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 3) if torch.cuda.is_available() else 0.0

    fold_metrics = {
        "fold": fold,
        "model_name": config["MODEL_NAME"],
        "best_f1": float(best_f1),
        "per_class_f1": [float(x) for x in best_per_class],
        "best_epoch_step": best_step_desc,
        "n_bad_steps": int(n_bad_steps),
        "minutes": float(fold_minutes),
        "peak_vram_gb": float(peak_vram),
    }
    with open(metrics_save_path, "w", encoding="utf-8") as f:
        json.dump(fold_metrics, f, indent=2)

    log_msg(
        f"[HOÀN THÀNH FOLD {fold}] Best Val Macro-F1: {best_f1:.4f} | "
        f"Thời gian: {fold_minutes:.1f} phút | VRAM: {peak_vram:.2f} GB",
        log_file,
    )

    if ckpt_path.exists():
        ckpt_path.unlink()
    if adapter_dir.exists():
        shutil.rmtree(adapter_dir, ignore_errors=True)
    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()


def main() -> None:
    script_t0 = time.time()
    parser = argparse.ArgumentParser(description="Huấn luyện LLM QLoRA K-Fold trên Kaggle.")
    parser.add_argument("--folds", nargs="+", type=int, required=True)
    parser.add_argument("--out_dir", type=str, default="/kaggle/working")
    parser.add_argument("--smoke", action="store_true")
    parser.add_argument("--resume_dir", type=str, default=None)
    args = parser.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    logs_dir = out_dir / "logs"
    logs_dir.mkdir(parents=True, exist_ok=True)

    config_file = out_dir / "config.json"
    if not config_file.exists():
        config_file = Path("config.json")
    with open(config_file, "r", encoding="utf-8") as f:
        config = json.load(f)

    # Hỗ trợ cả RESUME_DIRS (list) và RESUME_DIR (đơn)
    resume_dirs: List[str] = []
    if args.resume_dir:
        resume_dirs.append(args.resume_dir)
    if config.get("RESUME_DIR"):
        resume_dirs.append(str(config["RESUME_DIR"]))
    for rd in config.get("RESUME_DIRS", []) or []:
        if rd:
            resume_dirs.append(str(rd))

    for rdir in resume_dirs:
        res_path = Path(rdir)
        if res_path.exists():
            log_msg(f"[*] Sao chép file fold có sẵn từ: {res_path}...")
            for f in res_path.rglob("*_fold*.*"):
                dest = out_dir / f.name
                if not dest.exists():
                    shutil.copy2(f, dest)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log_msg(f"[*] Thiết bị: {device} | CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'All')}")

    model_name = config["MODEL_NAME"]
    model_dir = find_model_dir(model_name)
    log_msg(f"[*] Đã tìm thấy model '{model_name}' tại: {model_dir}")

    tokenizer, pad_id = setup_tokenizer_and_pad(model_dir)
    log_msg(f"[*] Tokenizer sẵn sàng: pad_token='{tokenizer.pad_token}' (id={pad_id}), eos_id={tokenizer.eos_token_id}")

    train_df = pd.read_csv(find_file("train.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
    test_df = pd.read_csv(find_file("test.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
    folds_df = pd.read_csv(find_file("folds.csv"), dtype={"id": str})

    train_df = train_df.merge(folds_df, on="id", how="left", validate="1:1")
    assert train_df["fold"].notna().all(), "folds.csv không khớp 1:1 với train.csv!"
    train_df["fold"] = train_df["fold"].astype(int)
    train_df["label_id"] = train_df["label"].map(LABEL2ID)

    train_bodies = [f"lang: {r['language']} | {clean_text_safety(r['text'])}" for _, r in train_df.iterrows()]
    test_bodies = [f"lang: {r['language']} | {clean_text_safety(r['text'])}" for _, r in test_df.iterrows()]

    max_len = int(config["MAX_LEN"])
    log_msg(f"[*] Tokenize Train & Test với Instruction + Head/Tail (MAX_LEN={max_len})...")
    train_input_ids = prepare_llm_input_ids(train_bodies, tokenizer, max_len=max_len, pad_id=pad_id)
    test_input_ids = prepare_llm_input_ids(test_bodies, tokenizer, max_len=max_len, pad_id=pad_id)

    for fold in args.folds:
        fold_log_file = logs_dir / f"fold{fold}.log"
        train_single_fold(
            fold=fold,
            config=config,
            train_df=train_df,
            test_df=test_df,
            train_input_ids=train_input_ids,
            test_input_ids=test_input_ids,
            model_dir=model_dir,
            pad_id=pad_id,
            device=device,
            script_t0=script_t0,
            is_smoke=args.smoke,
            out_dir=out_dir,
            log_file=fold_log_file,
        )


if __name__ == "__main__":
    main()


In [ ]:
# =============================================================================
# 🧪 3. SMOKE TEST (CHẠY THỬ NHANH FOLD 0: 256 TRAIN / 128 VAL / 64 TEST, 30 STEPS)
# =============================================================================
import subprocess
import sys

if SMOKE_TEST:
    print("[*] Đang chạy Smoke Test cho QLoRA trên Fold 0 (256 train, 128 val, 64 test, 30 opt steps)...", flush=True)
    cmd = [sys.executable, "train_llm.py", "--folds", "0", "--smoke", "--out_dir", OUT_DIR]
    res = subprocess.run(cmd)
    if res.returncode != 0:
        raise RuntimeError(f"Smoke test thất bại với mã lỗi {res.returncode}! Hãy kiểm tra log ở trên.")
    print("\n[+] SMOKE TEST HOÀN TẤT THÀNH CÔNG!", flush=True)
    print("[+] Hãy copy log báo cáo VRAM / tokens/sec / ETA ở trên gửi lại để qua Gate Smoke Test.", flush=True)
else:
    print("[*] SMOKE_TEST = False. Bỏ qua cell Smoke Test để chạy huấn luyện chính thức ở Cell 4.", flush=True)


In [ ]:
# =============================================================================
# 🚀 4. KHỞI CHẠY HUẤN LUYỆN SONG SONG 2 GPU T4 THEO FOLDS_TO_RUN
# =============================================================================
import os
import subprocess
import sys
import time
from pathlib import Path
import torch

if SMOKE_TEST:
    print("[!] SMOKE_TEST = True: Bỏ qua cell huấn luyện chính thức.", flush=True)
else:
    n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f"[*] Phát hiện {n_gpus} GPU khả dụng. FOLDS_TO_RUN = {FOLDS_TO_RUN}", flush=True)

    log_dir = Path(OUT_DIR) / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)

    processes = []
    log_files = []
    log_paths = []
    extra_args = ["--resume_dir", str(RESUME_DIR)] if RESUME_DIR else []

    if n_gpus >= 2:
        for gpu_id, fold_list in FOLDS_TO_RUN.items():
            if not fold_list:
                continue
            folds_str = [str(f) for f in fold_list]
            log_path = log_dir / f"gpu{gpu_id}_launch.log"
            lf = open(log_path, "w", encoding="utf-8", buffering=1)
            log_files.append(lf)
            log_paths.append(log_path)
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_id)}
            cmd = [sys.executable, "train_llm.py", "--folds", *folds_str, "--out_dir", OUT_DIR, *extra_args]
            p = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)
            processes.append(p)
            print(f"[+] Đã khởi chạy GPU {gpu_id} (PID: {p.pid}) cho các fold {folds_str}", flush=True)
    else:
        all_folds = sorted({int(f) for flist in FOLDS_TO_RUN.values() for f in flist})
        if all_folds:
            folds_str = [str(f) for f in all_folds]
            log_path = log_dir / "gpu0_launch.log"
            lf = open(log_path, "w", encoding="utf-8", buffering=1)
            log_files.append(lf)
            log_paths.append(log_path)
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"} if n_gpus == 1 else {**os.environ}
            cmd = [sys.executable, "train_llm.py", "--folds", *folds_str, "--out_dir", OUT_DIR, *extra_args]
            p = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)
            processes.append(p)
            print(f"[+] Đã khởi chạy tuần tự (PID: {p.pid}) cho các fold {folds_str}", flush=True)

    start_time = time.time()
    while any(p.poll() is None for p in processes):
        time.sleep(120)
        elapsed_min = (time.time() - start_time) / 60.0
        print(f"\n--- [Thời gian đã chạy: {elapsed_min:.1f} phút] ---", flush=True)
        for i, (proc, lp) in enumerate(zip(processes, log_paths)):
            status = "Đang chạy" if proc.poll() is None else f"Hoàn thành (mã {proc.poll()})"
            print(f"> Process {i} (PID {proc.pid}) - Trạng thái: {status}", flush=True)
            if lp.exists():
                try:
                    lines = lp.read_text(encoding="utf-8", errors="replace").strip().splitlines()
                    print("  " + "\n  ".join(lines[-3:]) if lines else "  ...", flush=True)
                except Exception as e:
                    print(f"  (Chưa đọc được log: {e})", flush=True)
        sys.stdout.flush()

    for lf in log_files:
        lf.close()

    for i, (proc, lp) in enumerate(zip(processes, log_paths)):
        if proc.returncode != 0:
            lines = lp.read_text(encoding="utf-8", errors="replace").strip().splitlines()
            raise RuntimeError(f"Process {i} thất bại (mã {proc.returncode}):\n" + "\n".join(lines[-50:]))

    print("\n[+] HOÀN TẤT CÁC FOLD ĐƯỢC PHÂN CÔNG TRONG PHIÊN NÀY!", flush=True)


In [ ]:
# =============================================================================
# 📊 5. GHÉP OOF (HỖ TRỢ PARTIAL FOLDS & RESUME_DIRS), MULTIPLIER & SUBMISSION
# =============================================================================
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import f1_score

if SMOKE_TEST:
    print("[!] SMOKE_TEST = True: Bỏ qua cell ghép OOF và sinh submission.", flush=True)
else:
    out_dir = Path(OUT_DIR)

    # 1. Gom các file *_fold*.* từ RESUME_DIRS (nhiều tài khoản/version)
    all_resume_dirs = list(RESUME_DIRS) if RESUME_DIRS else []
    if RESUME_DIR:
        all_resume_dirs.append(RESUME_DIR)
    for rdir in all_resume_dirs:
        rp = Path(rdir)
        if rp.exists():
            for f in rp.rglob("*_fold*.*"):
                dest = out_dir / f.name
                if not dest.exists():
                    shutil.copy2(f, dest)
                    print(f"[*] Đã gộp {f.name} từ {rp}", flush=True)

    search_dirs = [Path("/kaggle/input"), Path("data"), Path(".")]
    def find_file_local(name):
        for base in search_dirs:
            if base.exists():
                for f in sorted(base.rglob(name)):
                    if f.is_file():
                        return f
        raise FileNotFoundError(f"Không tìm thấy {name}")

    train_df = pd.read_csv(find_file_local("train.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
    test_df = pd.read_csv(find_file_local("test.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
    folds_df = pd.read_csv(find_file_local("folds.csv"), dtype={"id": str})
    train_df = train_df.merge(folds_df, on="id", how="left", validate="1:1")
    assert train_df["fold"].notna().all()

    VALID_LABELS = ("BENIGN", "JAILBREAK", "PROMPT_INJECTION")
    LABEL2ID = {"BENIGN": 0, "JAILBREAK": 1, "PROMPT_INJECTION": 2}
    ID2LABEL = {0: "BENIGN", 1: "JAILBREAK", 2: "PROMPT_INJECTION"}
    y_true = train_df["label"].map(LABEL2ID).values

    # 2. Kiểm tra các fold hiện có (hỗ trợ chạy từng phần 1-5 fold)
    available_folds = []
    missing_folds = []
    for k in range(5):
        if (out_dir / f"oof_fold{k}.npy").exists() and (out_dir / f"val_idx_fold{k}.npy").exists() and (out_dir / f"test_fold{k}.npy").exists():
            available_folds.append(k)
        else:
            missing_folds.append(k)

    if not available_folds:
        raise RuntimeError("Không tìm thấy bất kỳ fold nào hoàn thành!")

    print(f"[*] Các fold đã hoàn thành ({len(available_folds)}/5): {available_folds}", flush=True)
    if missing_folds:
        print(f"[!] LƯU Ý: Còn thiếu các fold {missing_folds}. Đang tính OOF & Test trung bình trên các fold đã có: {available_folds}", flush=True)

    oof_matrix = np.full((len(train_df), 3), np.nan, dtype=np.float32)
    per_fold_metrics = {}
    test_fold_probs = []

    for k in available_folds:
        oof_k = np.load(out_dir / f"oof_fold{k}.npy")
        val_idx_k = np.load(out_dir / f"val_idx_fold{k}.npy")
        oof_matrix[val_idx_k] = oof_k
        test_fold_probs.append(np.load(out_dir / f"test_fold{k}.npy"))
        m_path = out_dir / f"metrics_fold{k}.json"
        if m_path.exists():
            with open(m_path, "r", encoding="utf-8") as f:
                per_fold_metrics[f"fold_{k}"] = json.load(f)

    valid_mask = ~np.isnan(oof_matrix[:, 0])
    y_eval = y_true[valid_mask]
    oof_eval = oof_matrix[valid_mask]
    df_eval = train_df.loc[valid_mask].reset_index(drop=True)

    def optimize_class_multipliers(y_idx, proba):
        base_preds = proba.argmax(axis=1)
        base_f1 = float(f1_score(y_idx, base_preds, average="macro", zero_division=0))
        best_f1 = base_f1
        best_m = np.ones(3, dtype=float)
        grid_vals = np.linspace(0.3, 3.0, 28)
        for m1 in grid_vals:
            for m2 in grid_vals:
                cand = np.array([1.0, m1, m2], dtype=float)
                score = float(f1_score(y_idx, (proba * cand).argmax(axis=1), average="macro", zero_division=0))
                if score > best_f1:
                    best_f1 = score
                    best_m = cand

        def _loss(params):
            if np.any(params <= 0.05) or np.any(params > 10.0):
                return 1.0
            m_vec = np.concatenate([[1.0], params])
            return -float(f1_score(y_idx, (proba * m_vec).argmax(axis=1), average="macro", zero_division=0))

        res = minimize(_loss, x0=best_m[1:], method="Nelder-Mead", options={"maxiter": 300, "xatol": 1e-3, "fatol": 1e-5})
        if -float(res.fun) > best_f1:
            best_f1 = -float(res.fun)
            best_m = np.concatenate([[1.0], res.x])
        return best_m, base_f1, best_f1

    best_multipliers, base_f1, global_tuned_f1 = optimize_class_multipliers(y_eval, oof_eval)
    oof_preds_base = oof_eval.argmax(axis=1)
    oof_preds_tuned = (oof_eval * best_multipliers).argmax(axis=1)

    cls_f1_base = f1_score(y_eval, oof_preds_base, average=None, zero_division=0)
    cls_f1_tuned = f1_score(y_eval, oof_preds_tuned, average=None, zero_division=0)

    # Đánh giá riêng cụm khó pt & 500-2000 ký tự & B/J
    char_lens = df_eval["text"].astype(str).str.len().values
    pt_bj_mask = (df_eval["language"].values == "pt") & (char_lens >= 500) & (char_lens <= 2000) & np.isin(y_eval, [0, 1])
    pt_bj_f1 = float(f1_score(y_eval[pt_bj_mask], oof_preds_tuned[pt_bj_mask], labels=[0, 1], average="macro", zero_division=0)) if pt_bj_mask.any() else 0.0
    pt_bj_errors = int((y_eval[pt_bj_mask] != oof_preds_tuned[pt_bj_mask]).sum()) if pt_bj_mask.any() else 0

    per_language_f1 = {}
    for lang in ("pt", "fr", "it", "en"):
        m = (df_eval["language"] == lang).values
        if m.any():
            per_language_f1[lang] = float(f1_score(y_eval[m], oof_preds_tuned[m], average="macro", zero_division=0))

    print(f"\n[+] OOF ({ int(valid_mask.sum()):,} mẫu từ folds {available_folds}) Macro-F1 (Base):   {base_f1:.4f}", flush=True)
    print(f"[+] OOF ({ int(valid_mask.sum()):,} mẫu từ folds {available_folds}) Macro-F1 (Global): {global_tuned_f1:.4f} (m={np.round(best_multipliers, 4).tolist()})", flush=True)
    print(f"[+] Cụm khó (pt, 500-2000 ký tự, B/J): F1 = {pt_bj_f1:.4f} | Lỗi: {pt_bj_errors}/{int(pt_bj_mask.sum())}", flush=True)

    final_test_probs = np.mean(test_fold_probs, axis=0)
    final_test_preds = (final_test_probs * best_multipliers).argmax(axis=1)

    sub_df = pd.DataFrame({"id": test_df["id"], "label": [ID2LABEL[int(i)] for i in final_test_preds]})
    sub_df.to_csv(out_dir / "submission.csv", index=False)
    np.save(out_dir / "oof_probs_mistral.npy", oof_matrix)
    np.save(out_dir / "test_probs_mistral.npy", final_test_probs)

    metrics_summary = {
        "model_name": MODEL_NAME,
        "available_folds": available_folds,
        "missing_folds": missing_folds,
        "n_oof_evaluated": int(valid_mask.sum()),
        "oof_macro_f1_base": float(base_f1),
        "oof_macro_f1_global_tuned": float(global_tuned_f1),
        "oof_macro_f1_tuned": float(global_tuned_f1),
        "multipliers": best_multipliers.tolist(),
        "per_class_f1_base": {VALID_LABELS[i]: float(cls_f1_base[i]) for i in range(3)},
        "per_class_f1_tuned": {VALID_LABELS[i]: float(cls_f1_tuned[i]) for i in range(3)},
        "per_language_f1": per_language_f1,
        "pt_500_2000_bj_f1": pt_bj_f1,
        "pt_500_2000_bj_errors": pt_bj_errors,
        "per_fold": per_fold_metrics,
    }
    with open(out_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_summary, f, indent=2, ensure_ascii=False)
    print("[+] Đã lưu submission.csv, oof_probs_mistral.npy, test_probs_mistral.npy và metrics.json!", flush=True)


In [ ]:
# =============================================================================
# 🛡️ 6. KIỂM ĐỊNH CHẤT LƯỢNG SUBMISSION (SANITY CHECK)
# =============================================================================
import json
from pathlib import Path
import pandas as pd

if SMOKE_TEST:
    print("[!] SMOKE_TEST = True: Bỏ qua sanity check của submission.", flush=True)
else:
    out_dir = Path(OUT_DIR)
    sub_path = out_dir / "submission.csv"
    assert sub_path.exists(), "Tệp submission.csv không tồn tại!"
    sub_df = pd.read_csv(sub_path, dtype={"id": str})

    assert len(sub_df) == 6000, f"Số dòng phải đúng 6,000, thực tế: {len(sub_df)}"
    assert list(sub_df.columns) == ["id", "label"], f"Cột không hợp lệ: {list(sub_df.columns)}"
    assert not sub_df.isna().any().any(), "Submission chứa giá trị NaN!"
    assert not sub_df["id"].duplicated().any(), "Submission chứa ID trùng lặp!"

    VALID_LABELS = {"BENIGN", "JAILBREAK", "PROMPT_INJECTION"}
    assert set(sub_df["label"].unique()).issubset(VALID_LABELS)

    dist = sub_df["label"].value_counts(normalize=True) * 100
    counts = sub_df["label"].value_counts()
    pi_pct = float(dist.get("PROMPT_INJECTION", 0.0))

    print("\n" + "=" * 60, flush=True)
    print("📋 PHÂN PHỐI DỰ ĐOÁN TRÊN TẬP TEST (6,000 DÒNG):", flush=True)
    print("=" * 60, flush=True)
    for lbl in ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]:
        print(f"  - {lbl:<18}: {counts.get(lbl, 0):>5,} dòng ({dist.get(lbl, 0.0):>5.2f}%)", flush=True)

    if pi_pct < 5.0 or pi_pct > 10.0:
        print(f"[!] CẢNH BÁO: PROMPT_INJECTION ({pi_pct:.2f}%) nằm ngoài khoảng 5%-10%!", flush=True)
    else:
        print(f"[+] Tỷ lệ PROMPT_INJECTION ({pi_pct:.2f}%) nằm chuẩn trong khoảng 5%-10%.", flush=True)
